In [79]:
import importlib
import edgar_client
importlib.reload(edgar_client)  # force reload the updated file
import filing_parser
importlib.reload(filing_parser)

<module 'filing_parser' from '/Users/vivianjohnson/Desktop/rag_pipeline_project/sec_fred_rag_app/filing_parser.py'>

In [80]:
DEFAULT_TICKERS = [
    "AAPL",   # Apple
    # "MSFT",   # Microsoft
    # "JPM",    # JPMorgan Chase
    # "BAC",    # Bank of America
    # "XOM",    # ExxonMobil
    # "JNJ",    # Johnson & Johnson
    # "AMZN",   # Amazon
    # "NVDA"    # Nvidia
]

DEFAULT_FORM_TYPE = "10-K"
DEFAULT_LIMIT = 3          # filings per ticker
FRED_START_DATE = "2010-01-01"

In [90]:
def run_sec_ingestion(
    tickers: list[str],
    form_type: str="10-K"
):
    """Full SEC ingestion: fetch → parse → embed."""
    edgar = edgar_client.EdgarClient()
    parser = filing_parser.FilingParser()

    print(f"\n{'='*60}")
    print(f"SEC Ingestion: {len(tickers)} tickers")
    print(f"{'='*60}\n")

    for ticker in tickers:
        # 1. get CIK and company info
        cik = edgar.get_cik(ticker)
        print(cik)

        # 2. get filing list 
        filings = edgar.get_filings(cik=cik)
        print(filings)
        # get filing text 
        for i in range(0, 1):
            f = filings[i]
            text = edgar.get_filing_text(f)
            # 3. parse doc 
            sections = parser.parse(text)
            # print(sections)
            for s in sections.values(): 
                chunks = filing_parser.chunk_text(s)
                print(chunks[0:10])

In [91]:
edgar = edgar_client.EdgarClient()
filings = run_sec_ingestion(DEFAULT_TICKERS)


SEC Ingestion: 1 tickers

[EDGAR] AAPL -> CIK 0000320193 (Apple Inc.)
0000320193
[EDGAR] Found 5 10-K filings for CIK 0000320193
[{'cik': '0000320193', 'accession_number': '0000320193-25-000079', 'filing_date': '2025-10-31', 'form_type': '10-K', 'primary_document': 'aapl-20250927.htm', 'document_url': 'https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/aapl-20250927.htm', 'index_url': 'https://www.sec.gov/Archives/edgar/data/320193/000032019325000079/'}, {'cik': '0000320193', 'accession_number': '0000320193-24-000123', 'filing_date': '2024-11-01', 'form_type': '10-K', 'primary_document': 'aapl-20240928.htm', 'document_url': 'https://www.sec.gov/Archives/edgar/data/320193/000032019324000123/aapl-20240928.htm', 'index_url': 'https://www.sec.gov/Archives/edgar/data/320193/000032019324000123/'}, {'cik': '0000320193', 'accession_number': '0000320193-23-000106', 'filing_date': '2023-11-03', 'form_type': '10-K', 'primary_document': 'aapl-20230930.htm', 'document_url': 'https:/